<a href="https://colab.research.google.com/github/mehmetbozdemir24/Magibu/blob/main/01_Data_Preparation/Yemek_Tarifleri_WEB_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cwd = "/content/drive/MyDrive/Colab Notebooks/Magibu/"

In [ ]:
# --- Hücre 1: Ortam hazırlığı ---
from google.colab import drive
import os

# Drive bağla (zaten bağlıysa uyarı verir, sorun değil)
drive.mount('/content/drive')

# Çalışma dizini
cwd = "/content/drive/MyDrive/Colab Notebooks/Magibu/"

# Proje için alt klasörler
RAW_DIR = os.path.join(cwd, "data", "raw")        # ham HTML / ara çıktılar
OUT_DIR = os.path.join(cwd, "data", "processed")  # işlenmiş dataset

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Gerekli kütüphaneler
!pip -q install requests beautifulsoup4 lxml

import requests
from bs4 import BeautifulSoup

print("cwd var mı? ->", os.path.isdir(cwd))
print("RAW_DIR:", RAW_DIR)
print("OUT_DIR:", OUT_DIR)
print("requests:", requests.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd var mı? -> True
RAW_DIR: /content/drive/MyDrive/Colab Notebooks/Magibu/data/raw
OUT_DIR: /content/drive/MyDrive/Colab Notebooks/Magibu/data/processed
requests: 2.32.4


In [ ]:
# --- Hücre 2b: Cloudflare'ı cloudscraper ile deneme ---
!pip -q install cloudscraper

import cloudscraper

scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

TEST_URL = "https://www.nefisyemektarifleri.com/mercimek-corbasi-2/"

resp = scraper.get(TEST_URL, timeout=30)
print("Status code:", resp.status_code)
print("Gelen içerik boyutu (byte):", len(resp.content))

# Başlık kontrolü
soup = BeautifulSoup(resp.text, "lxml")
title = soup.find("title")
print("Sayfa başlığı:", title.get_text(strip=True) if title else "BULUNAMADI")

# JSON-LD var mı? (tarif sitelerinde malzeme/adım genelde burada gömülü olur)
ld = soup.find_all("script", type="application/ld+json")
print("JSON-LD blok sayısı:", len(ld))

# Başarılıysa ham HTML'i güncelle
if resp.status_code == 200 and "Just a moment" not in resp.text:
    with open(os.path.join(RAW_DIR, "test_page.html"), "w", encoding="utf-8") as f:
        f.write(resp.text)
    print("✓ Sayfa başarıyla çekildi ve kaydedildi.")
else:
    print("✗ Hâlâ engelleniyor — Playwright'a veya başka siteye geçelim.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 7.1 MB/s eta 0:00:00
Status code: 200
Gelen içerik boyutu (byte): 259384
Sayfa başlığı: Sebzeli Mercimek Çorbası Tarifi - Nefis Yemek Tarifleri
JSON-LD blok sayısı: 1
✓ Sayfa başarıyla çekildi ve kaydedildi.


In [ ]:
# --- Hücre 3: JSON-LD içeriğini inceleme ---
import json

with open(os.path.join(RAW_DIR, "test_page.html"), "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "lxml")
ld_scripts = soup.find_all("script", type="application/ld+json")

data = json.loads(ld_scripts[0].string)

# JSON-LD bazen tek obje, bazen liste, bazen @graph içinde gelir — hepsini düzleştir
def iter_nodes(obj):
    if isinstance(obj, list):
        for x in obj:
            yield from iter_nodes(x)
    elif isinstance(obj, dict):
        if "@graph" in obj:
            yield from iter_nodes(obj["@graph"])
        else:
            yield obj

# Recipe tipindeki düğümü bul
recipe = None
for node in iter_nodes(data):
    t = node.get("@type", "")
    if t == "Recipe" or (isinstance(t, list) and "Recipe" in t):
        recipe = node
        break

if recipe is None:
    print("✗ Recipe tipi bulunamadı. Mevcut anahtarlar:")
    print(list(data.keys()) if isinstance(data, dict) else type(data))
else:
    print("✓ Recipe bulundu. Mevcut alanlar:\n")
    for k in recipe.keys():
        v = recipe[k]
        # Değeri kısaca özetle
        if isinstance(v, list):
            preview = f"[liste, {len(v)} eleman] örnek: {v[0] if v else '-'}"
        elif isinstance(v, (dict,)):
            preview = f"{{sözlük}} anahtarlar: {list(v.keys())}"
        else:
            preview = str(v)[:100]
        print(f"  {k}: {preview}")

✗ Recipe tipi bulunamadı. Mevcut anahtarlar:
['@context', '@type', 'name', 'url', 'potentialAction']


In [ ]:
# --- Hücre 4: Malzeme/adım bloklarını HTML içinde bulma ---
with open(os.path.join(RAW_DIR, "test_page.html"), "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "lxml")

# 1) "malzeme" ve "yapılış" başlıklarına yakın yapıları arayalım
print("=== 'malzeme' geçen id/class'lar ===")
for tag in soup.find_all(True):
    attrs = " ".join(str(v) for v in tag.attrs.values()).lower()
    if "malzeme" in attrs or "ingredient" in attrs:
        print(f"<{tag.name}> attrs={tag.attrs}")

print("\n=== 'tarif' / 'yapılış' / 'adim' / 'step' / 'instruction' geçen id/class'lar ===")
for tag in soup.find_all(True):
    attrs = " ".join(str(v) for v in tag.attrs.values()).lower()
    if any(k in attrs for k in ["yapilis", "yapılış", "adim", "step", "instruction", "recipe-instruction"]):
        print(f"<{tag.name}> attrs={tag.attrs}")

# 2) Bir malzeme adını (örn. 'mercimek') içeren en küçük tag'i bulup üst yapısını göster
print("\n=== 'mercimek' geçen küçük tag'ler (ilk 5) ===")
count = 0
for tag in soup.find_all(["li", "span", "p", "td"]):
    txt = tag.get_text(" ", strip=True)
    if "mercimek" in txt.lower() and len(txt) < 80:
        parent = tag.parent
        print(f"<{tag.name} class={tag.get('class')}> -> '{txt}'  | parent<{parent.name} class={parent.get('class')}>")
        count += 1
        if count >= 5:
            break

=== 'malzeme' geçen id/class'lar ===
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<li> attrs={'itemprop': 'recipeIngredient'}
<div> attrs={'class': ['no-print'], 'id': 'malzemeler_yani_banner-container'}
<div> attrs={'class': ['no-print', 'nokta'], 'data-nokta-zone': '153193', 'id': 'malzemeler_yani_banner'}

=== 'tarif' / 'yapılış' / 'adim' / 'step' / 'instruction' geçen id/class'lar ===
<ol> attrs={'class': ['recipe-instructions'], 'itemprop': 'recipeInstructions'}
<div> attrs={'data-jshref': 'https://www.nefisyemektarifleri.com/sebzeli-mercimek-corbasi-yapilisi/', 'data-jshref-for': 'mobile'}
<a> attrs={'href': 'https://www.nefisyemektarifleri.com/sebzeli-mercimek-corbasi-yapilisi/

In [ ]:
# --- Hücre 5: Tek tarifi yapılandırılmış sözlüğe çeviren parser ---

def parse_recipe(html, url=None):
    soup = BeautifulSoup(html, "lxml")

    # Başlık
    name_tag = soup.find(attrs={"itemprop": "name"})
    if name_tag:
        title = name_tag.get_text(strip=True)
    else:
        t = soup.find("title")
        title = t.get_text(strip=True) if t else None

    # Malzemeler
    malzemeler = [
        li.get_text(" ", strip=True)
        for li in soup.find_all("li", attrs={"itemprop": "recipeIngredient"})
        if li.get_text(strip=True)
    ]

    # Adımlar
    adimlar = []
    ol = soup.find("ol", attrs={"itemprop": "recipeInstructions"})
    if ol is None:
        ol = soup.find("ol", class_="recipe-instructions")
    if ol:
        adimlar = [
            li.get_text(" ", strip=True)
            for li in ol.find_all("li")
            if li.get_text(strip=True)
        ]

    # Süre (microdata: cookTime / totalTime, çoğu ISO-8601 "PT30M" formatında)
    sure = None
    for prop in ["cookTime", "totalTime", "prepTime"]:
        tag = soup.find(attrs={"itemprop": prop})
        if tag:
            sure = tag.get("content") or tag.get_text(strip=True)
            if sure:
                break

    # Porsiyon
    porsiyon = None
    y = soup.find(attrs={"itemprop": "recipeYield"})
    if y:
        porsiyon = y.get("content") or y.get_text(" ", strip=True)

    return {
        "baslik": title,
        "url": url,
        "malzemeler": malzemeler,
        "adimlar": adimlar,
        "sure": sure,
        "porsiyon": porsiyon,
    }


# Test edelim
with open(os.path.join(RAW_DIR, "test_page.html"), "r", encoding="utf-8") as f:
    html = f.read()

recipe = parse_recipe(html, url=TEST_URL)

print("Başlık :", recipe["baslik"])
print("Süre   :", recipe["sure"])
print("Porsiyon:", recipe["porsiyon"])
print(f"\nMalzemeler ({len(recipe['malzemeler'])} adet):")
for m in recipe["malzemeler"]:
    print("  -", m)
print(f"\nAdımlar ({len(recipe['adimlar'])} adet):")
for i, a in enumerate(recipe["adimlar"], 1):
    print(f"  {i}. {a}")

Başlık : Yemek Tarifleri
Süre   : PT30M
Porsiyon: 5

Malzemeler (9 adet):
  - 1 su bardağı kırmızı mercimek
  - 1 adet orta boy kuru soğan
  - 1 adet havuç
  - 1 kahve fincanı pirinç
  - 1 adet orta boy patates
  - Su
  - Tuz
  - 1 yemek kaşığı tereyağı
  - 1 tatlı kaşığı tatlı kırmızı biber

Adımlar (9 adet):
  1. 1 su bardağı kırmızı mercimeği , 1 kahve fincanı pirinci derin bir kaba alarak iyice yıkayın.
  2. 1 adet orta boy soğanı soyup yıkayın.
  3. Patatesi soyup küçük parçalara ayırın.
  4. 1 adet havucu soyarak parçalara ayırın.
  5. Mercimek, pirinç, soğan, havuç ve patatesleri derin bir tencereye alın.
  6. Malzemelerin üzerini 3-4 parmak geçecek kadar suyla doldurun.
  7. Tüm malzemeler iyice yumuşayana dek pişirin. Tuzunu ayarlayın.
  8. Blenderle çorbayı pürüzsüz olacak şekilde ezin.
  9. 1 yemek kaşığı tereyağını ayrı bir kaba alarak kızdırın üzerine 1 tatlı kaşığı tatlı kırmızı biber ekleyin. Çorbanın üzerine gezdirin. Sıcak olarak servis edin. Afiyet olsun…


In [ ]:
# --- Hücre 6: Tarif URL'lerini sitemap'ten toplama ---
import re, time, json

TARGET = 1000  # hedef tarif sayısı

def fetch(url, tries=3, sleep=2):
    for _ in range(tries):
        try:
            r = scraper.get(url, timeout=30)
            if r.status_code == 200:
                return r.text
        except Exception as e:
            print("  istek hatası:", e)
        time.sleep(sleep)
    return None

def extract_locs(xml):
    return re.findall(r"<loc>\s*(.*?)\s*</loc>", xml or "", flags=re.I)

# 1) Sitemap index'i bul
index_xml = None
for u in ["https://www.nefisyemektarifleri.com/sitemap_index.xml",
          "https://www.nefisyemektarifleri.com/sitemap.xml"]:
    txt = fetch(u)
    if txt and "<loc>" in txt.lower():
        print("✓ Sitemap index:", u)
        index_xml = txt
        break

sub_sitemaps = [s for s in extract_locs(index_xml) if s.endswith(".xml")]
print("Alt sitemap sayısı:", len(sub_sitemaps))

# 2) Alt sitemap'leri gez, tarif URL'lerini topla ("-tarifi/" ile bitenler)
recipe_urls, seen = [], set()
for sm in sub_sitemaps:
    if len(recipe_urls) >= TARGET:
        break
    locs = extract_locs(fetch(sm))
    added = 0
    for u in locs:
        if u.endswith("-tarifi/") and u not in seen:
            seen.add(u); recipe_urls.append(u); added += 1
            if len(recipe_urls) >= TARGET:
                break
    if added:
        print(f"  {sm.split('/')[-1]} -> +{added} (toplam {len(recipe_urls)})")
    time.sleep(1)

print("\nToplam tarif URL:", len(recipe_urls))
print("Örnekler:")
for u in recipe_urls[:5]:
    print("  ", u)

# Kaydet
url_path = os.path.join(RAW_DIR, "recipe_urls.json")
with open(url_path, "w", encoding="utf-8") as f:
    json.dump(recipe_urls, f, ensure_ascii=False, indent=2)
print("\n✓ URL listesi kaydedildi ->", url_path)

✓ Sitemap index: https://www.nefisyemektarifleri.com/sitemap.xml
Alt sitemap sayısı: 635
  sitemap-pt-post-2026-07.xml -> +2 (toplam 2)
  sitemap-pt-post-2026-06.xml -> +5 (toplam 7)
  sitemap-pt-post-2026-05.xml -> +1 (toplam 8)
  sitemap-pt-post-2026-04.xml -> +1 (toplam 9)
  sitemap-pt-post-2026-03.xml -> +1 (toplam 10)
  sitemap-pt-post-2026-01.xml -> +1 (toplam 11)
  sitemap-pt-post-2025-12.xml -> +1 (toplam 12)
  sitemap-pt-post-2025-07.xml -> +2 (toplam 14)
  sitemap-pt-post-2025-06.xml -> +2 (toplam 16)
  sitemap-pt-post-2025-03.xml -> +2 (toplam 18)
  sitemap-pt-post-2025-02.xml -> +3 (toplam 21)
  sitemap-pt-post-2025-01.xml -> +1 (toplam 22)
  sitemap-pt-post-2024-12.xml -> +2 (toplam 24)
  sitemap-pt-post-2024-11.xml -> +2 (toplam 26)
  sitemap-pt-post-2024-10.xml -> +2 (toplam 28)
  sitemap-pt-post-2024-09.xml -> +2 (toplam 30)
  sitemap-pt-post-2024-07.xml -> +5 (toplam 35)
  sitemap-pt-post-2024-06.xml -> +2 (toplam 37)
  sitemap-pt-post-2024-05.xml -> +1 (toplam 38)
  s

In [ ]:
# --- Hücre 7: Tüm tarifleri çek, parse et, conversation formatında kaydet ---
import json, time, os

# Ayarlar
LIMIT = None          # ÖNCE 10 ile test et; sonra None yapıp tamamını çek
SLEEP = 1.0         # istekler arası bekleme (siteyi yormamak için)
CKPT_EVERY = 50     # kaç tarifte bir diske yaz

url_path = os.path.join(RAW_DIR, "recipe_urls.json")
with open(url_path, "r", encoding="utf-8") as f:
    recipe_urls = json.load(f)

urls = recipe_urls if LIMIT is None else recipe_urls[:LIMIT]

out_path   = os.path.join(OUT_DIR, "recipe_dataset.jsonl")   # her satır 1 örnek
state_path = os.path.join(RAW_DIR, "progress.json")          # kaldığı yeri tutar

# Kaldığı yerden devam
done_urls = set()
if os.path.exists(state_path):
    with open(state_path, "r", encoding="utf-8") as f:
        done_urls = set(json.load(f))
    print(f"Önceki ilerleme bulundu: {len(done_urls)} tarif zaten işlenmiş.")

def title_from_html(soup, fallback_url):
    # Sayfa başlığından temiz tarif adı çıkar
    t = soup.find("title")
    if t:
        txt = t.get_text(strip=True)
        # "... Tarifi - Nefis Yemek Tarifleri" -> "... Tarifi"
        txt = txt.split(" - ")[0].strip()
        return txt
    return fallback_url.rstrip("/").split("/")[-1].replace("-", " ")

def build_example(recipe):
    """parse_recipe çıktısını doğal conversation formatına çevirir."""
    ad = recipe["baslik"]

    # 1. KULLANICI MESAJI: Doğal bir soru soruyoruz (JSON kelimesini kaldırdık)
    user_msg = {
        "content": f"{ad} nasıl yapılır? Tarifini verebilir misin?",
        "images": None,
        "role": "user",
        "thinking": None,
        "tool_calls": None,
    }

    # 2. ASİSTAN MESAJI: JSON yerine okunaklı, şık bir metin oluşturuyoruz
    malzemeler_text = "\n".join([f"- {m}" for m in recipe["malzemeler"]])
    adimlar_text = "\n".join([f"{i+1}. {a}" for i, a in enumerate(recipe["adimlar"])])
    sure = recipe.get("sure", "Belirtilmemiş")
    porsiyon = recipe.get("porsiyon", "Belirtilmemiş")

    asistan_cevap = f"Tabii, işte nefis bir {ad} tarifi!\n\n"
    asistan_cevap += f"⏱️ **Süre:** {sure} | 🍽️ **Porsiyon:** {porsiyon}\n\n"
    asistan_cevap += f"**Malzemeler:**\n{malzemeler_text}\n\n"
    asistan_cevap += f"**Yapılışı:**\n{adimlar_text}\n\nŞimdiden afiyet olsun!"

    assistant_msg = {
        "content": asistan_cevap,  # Artık json.dumps yok, doğrudan metin veriyoruz
        "images": None,
        "role": "assistant",
        "thinking": None,
        "tool_calls": None,
    }

    return {"messages": [user_msg, assistant_msg], "source_url": recipe["url"]}

# Ana döngü
written, skipped, failed = 0, 0, 0
# Test modunda temiz başlamak istersen aşağıdaki 2 satırı bir kez elle çalıştırıp dosyaları sil.
mode = "a"  # append; checkpoint mantığı için

for i, url in enumerate(urls, 1):
    if url in done_urls:
        skipped += 1
        continue

    html = fetch(url)  # Hücre 6'daki fetch fonksiyonu
    if not html:
        failed += 1
        print(f"  ✗ [{i}] indirilemedi: {url}")
        continue

    recipe = parse_recipe(html, url=url)
    soup = BeautifulSoup(html, "lxml")
    recipe["baslik"] = title_from_html(soup, url)  # başlığı title'dan düzelt

    # Boş tarifleri ele (malzeme veya adım yoksa atla)
    if not recipe["malzemeler"] or not recipe["adimlar"]:
        failed += 1
        print(f"  ✗ [{i}] boş tarif (malzeme/adım yok): {url}")
        done_urls.add(url)
        continue

    example = build_example(recipe)
    with open(out_path, mode, encoding="utf-8") as f:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")
    written += 1
    done_urls.add(url)

    if i % 10 == 0:
        print(f"  [{i}/{len(urls)}] yazılan: {written}, atlanan: {skipped}, hatalı: {failed}")

    # Checkpoint
    if i % CKPT_EVERY == 0:
        with open(state_path, "w", encoding="utf-8") as f:
            json.dump(sorted(done_urls), f, ensure_ascii=False)

    time.sleep(SLEEP)

# Son checkpoint
with open(state_path, "w", encoding="utf-8") as f:
    json.dump(sorted(done_urls), f, ensure_ascii=False)

print(f"\n✓ Bitti. Yeni yazılan: {written}, atlanan(zaten vardı): {skipped}, hatalı: {failed}")
print("Dataset ->", out_path)

# İlk örneği göster
with open(out_path, "r", encoding="utf-8") as f:
    first = f.readline()
print("\n--- İlk örnek ---")
print(json.dumps(json.loads(first), ensure_ascii=False, indent=2))

Önceki ilerleme bulundu: 10 tarif zaten işlenmiş.
  [20/1000] yazılan: 10, atlanan: 10, hatalı: 0
  [30/1000] yazılan: 20, atlanan: 10, hatalı: 0
  [40/1000] yazılan: 30, atlanan: 10, hatalı: 0
  [50/1000] yazılan: 40, atlanan: 10, hatalı: 0
  [60/1000] yazılan: 50, atlanan: 10, hatalı: 0
  [70/1000] yazılan: 60, atlanan: 10, hatalı: 0
  [80/1000] yazılan: 70, atlanan: 10, hatalı: 0
  [90/1000] yazılan: 80, atlanan: 10, hatalı: 0
  [100/1000] yazılan: 90, atlanan: 10, hatalı: 0
  [110/1000] yazılan: 100, atlanan: 10, hatalı: 0
  [120/1000] yazılan: 110, atlanan: 10, hatalı: 0
  [130/1000] yazılan: 120, atlanan: 10, hatalı: 0
  [140/1000] yazılan: 130, atlanan: 10, hatalı: 0
  [150/1000] yazılan: 140, atlanan: 10, hatalı: 0
  [160/1000] yazılan: 150, atlanan: 10, hatalı: 0
  [170/1000] yazılan: 160, atlanan: 10, hatalı: 0
  [180/1000] yazılan: 170, atlanan: 10, hatalı: 0
  [190/1000] yazılan: 180, atlanan: 10, hatalı: 0
  [200/1000] yazılan: 190, atlanan: 10, hatalı: 0
  [210/1000] yazı

In [ ]:
import pandas as pd
import json

# Dosya yolu
file_path = "/content/drive/MyDrive/Colab Notebooks/Magibu/data/processed/recipe_dataset.jsonl"

kayitlar = []

# jsonl dosyasını satır satır okuyoruz
with open(file_path, "r", encoding="utf-8") as f:
    for satir in f:
        veri = json.loads(satir)

        # User ve Assistant mesajlarını ayıklayalım
        user_msg = ""
        assistant_msg = ""

        for msg in veri.get("messages", []):
            if msg.get("role") == "user":
                user_msg = msg.get("content", "")
            elif msg.get("role") == "assistant":
                assistant_msg = msg.get("content", "")

        # DataFrame için satır sözlüğünü oluştur
        kayitlar.append({
            "User_Prompt": user_msg,
            "Assistant_Response": assistant_msg,
            "Source_URL": veri.get("source_url", "")
        })

# DataFrame'i oluştur
df = pd.DataFrame(kayitlar)

# İlk 5 satırı göster
df.head()

,User_Prompt,Assistant_Response,Source_URL
0,Cold Brew Tarifi nasıl yapılır? Tarifini vereb...,"Tabii, işte nefis bir Cold Brew Tarifi tarifi!...",https://www.nefisyemektarifleri.com/cold-brew-...
1,Kahvaltılık Yeşil Zeytin Salatası Tarifi nasıl...,"Tabii, işte nefis bir Kahvaltılık Yeşil Zeytin...",https://www.nefisyemektarifleri.com/kahvaltili...
2,Yeşil Mercimekli Tavuklu Salata Tarifi nasıl y...,"Tabii, işte nefis bir Yeşil Mercimekli Tavuklu...",https://www.nefisyemektarifleri.com/yesil-merc...
3,30 Kişilik Aşure Tarifi nasıl yapılır? Tarifin...,"Tabii, işte nefis bir 30 Kişilik Aşure Tarifi ...",https://www.nefisyemektarifleri.com/30-kisilik...
4,Kıymalı Kabak Spagetti Tarifi nasıl yapılır? T...,"Tabii, işte nefis bir Kıymalı Kabak Spagetti T...",https://www.nefisyemektarifleri.com/kiymali-ka...


In [ ]:
df["User_Prompt"][9]

'Mayasız Patatesli Bazlama Tarifi nasıl yapılır? Tarifini verebilir misin?'

In [ ]:
print(df["Assistant_Response"][9])

Tabii, işte nefis bir Mayasız Patatesli Bazlama Tarifi tarifi!

⏱️ **Süre:** PT15M | 🍽️ **Porsiyon:** 5

**Malzemeler:**
- 2 adet patates
- 1 adet yumurta
- 1 paket kabartma tozu
- 1 tatlı kaşığı tuz
- Yarım çay kaşığı karabiber
- Yarım su bardağı kaşar peyniri
- Yarım demet maydanoz
- 2,5 su bardağı un
- Yarım su bardağı un
- Tereyağı

**Yapılışı:**
1. Patatesli bazlama için öncelikle kabuklarını soyup iri parçalar halinde doğradığımız patateslerimizi haşlayalım.
2. Haşladığımız patatesleri karıştırma kabına alalım ve güzelce ezelim. Ardından soğuması için bekletelim.
3. Patateslerin ilk sıcaklığı çıktıktan sonra üzerine yumurta, maydanoz ve kaşar peyniri ekleyelim. Bu aşamada patateslerin ilk sıcaklığının çıkmış olmasına dikkat etmelisiniz. Aksi halde yumurta pişebilir.
4. Un, kabartma tozu, tuz ve karabiberi de ekleyerek hamuru yoğuralım.
5. Toparladığımız hamuru 8 eşit parçaya ayırarak beze haline getirelim.
6. Un serptiğimiz tezgahta bezelerimizi açalım. Kolay yapışan bir hamur o 

In [ ]:
len(df)

996